In [9]:
import pandas as pd 
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [2]:
model1=SentenceTransformer("BAAI/bge-small-en-v1.5")
model2=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\anaconda3\envs\surprise\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PGCP-AI.STUDENTSDC\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\anaconda3\envs\surprise\l

In [5]:
sentences=[
    'He batted today in the cricket match',
    'Tommorow will be sunday',
    'She played excellently yesterday with many goals',
    'Today will be the event',
    'He performed well today'    
]

In [6]:
embeddings = model1.encode(sentences)
embeddings.shape

(5, 384)

In [7]:
similarities = model1.similarity(embeddings, embeddings)
print(similarities.shape)

torch.Size([5, 5])


In [8]:
similarities

tensor([[1.0000, 0.6240, 0.6345, 0.6313, 0.7848],
        [0.6240, 1.0000, 0.5359, 0.8091, 0.5897],
        [0.6345, 0.5359, 1.0000, 0.5578, 0.7114],
        [0.6313, 0.8091, 0.5578, 1.0000, 0.6245],
        [0.7848, 0.5897, 0.7114, 0.6245, 1.0000]])

In [14]:
df=pd.read_csv(r"F:\MACHINE LEARNING\SessionsMaterial\Netflix_TV_Shows_Movies\Netflix_TV_Shows_Movies\titles.csv")
df_ind_mov=df[(df['production_countries'].str.contains('IN'))&(df['type']=='MOVIE')&(df['release_year']>=1980)]
df_ind_mov

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
19,tm94651,Dostana,MOVIE,Two close friends decide to enter law enforcem...,1980,NaN,161,"['drama', 'comedy', 'crime', 'romance', 'action']",['IN'],NaN,tt0080653,2.1,25.0,3.980,4.9
56,tm721687,Vaashi,MOVIE,"Ebin Mathew, a budding lawyer ambitiously join...",1983,NaN,123,"['drama', 'thriller']",['IN'],NaN,tt13913068,6.7,388.0,3.790,NaN
65,tm52815,Disco Dancer,MOVIE,"Anil, a street singer, is humiliated and drive...",1982,NaN,106,"['drama', 'romance']",['IN'],NaN,tt0208903,6.4,1476.0,7.879,5.3
66,tm159912,Agneepath,MOVIE,"The movie depicts the life of a young boy, Vij...",1990,NaN,174,"['drama', 'action', 'crime']",['IN'],NaN,tt0098999,7.6,8947.0,4.062,6.5
71,tm157603,Dil,MOVIE,Raja lives a poor lifestyle along with his dad...,1990,NaN,172,"['drama', 'comedy', 'romance']",['IN'],NaN,tt0099429,6.6,5316.0,3.992,6.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5766,tm1004011,Time to Dance,MOVIE,When a ballroom dancer’s shot at a crucial tou...,2021,NaN,2,"['romance', 'drama']",['IN'],NaN,tt8622232,2.2,963.0,1.043,4.0
5788,tm987730,Tribhanga,MOVIE,"When her estranged mother falls into a coma, a...",2021,NaN,95,"['drama', 'family']",['IN'],NaN,tt11102314,6.2,2922.0,4.294,7.0
5795,tm988613,Madam Chief Minister,MOVIE,A political-drama where a young woman from a s...,2021,NaN,123,['drama'],['IN'],NaN,tt13773882,4.8,1769.0,2.301,6.7
5821,tm1110241,Kaaval,MOVIE,Thampan and Antony are a long time best friend...,2021,NaN,148,"['thriller', 'action', 'drama']",['IN'],NaN,tt11182984,5.1,1497.0,1.978,5.3


In [17]:
all_desc = list(df_ind_mov['description'])
out_embs=[]
for r in tqdm(all_desc):
    embs=model1.encode(r)
    out_embs.append(embs)
out_embs=np.array(out_embs)
out_embs.shape

100%|██████████| 570/570 [00:21<00:00, 27.13it/s]


(570, 384)

In [19]:
movie_idx=list(df_ind_mov['title']+ "-" + df_ind_mov['release_year'].astype(str))
df_embd=pd.DataFrame(out_embs,index=movie_idx)
df_embd.to_csv("movie-embds.csv")

In [21]:
query="college guys having fun and enjoying"
qy_emb=model1.encode([query])
sims=[]
for e in tqdm(out_embs):
    sims.append(model1.similarity(qy_emb,e).numpy()[0])
df_ind=df_ind_mov[['id','title','description','release_year']].copy()
df_ind.loc[:,'sims']=np.array(sims)
df_ind.sort_values('sims',ascending=False)[:10]

100%|██████████| 570/570 [00:00<00:00, 6570.12it/s]


,id,title,description,release_year,sims
5391,tm1132325,Don,A youngster who is still figuring out what his...,2022,0.654030
972,tm185797,Fukrey,The paths of four dream-chasing college friend...,2013,0.641161
434,tm61056,Dil Chahta Hai,Three inseparable childhood friends are just o...,2001,0.621702
513,tm54147,Dostana,Two straight guys who pretend to be a gay coup...,2008,0.605501
4565,tm471802,Sur Sapata,This story is about young boys who have capabi...,2019,0.605204
5154,tm1213009,Dear Friend,Four close friends who live together in Bangal...,2022,0.605137
1455,tm233843,Pyaar Ka Punchnama 2,"After falling in love, three roommates experie...",2015,0.600313
492,tm88366,Main Hoon Na,An army major goes undercover as a college stu...,2004,0.585556
1656,tm232947,Brahman Naman,This homage to 1980s teen sex comedies follows...,2016,0.579227
4543,tm838022,Upstarts,"Hoping to do good while making millions, three...",2019,0.576946


In [24]:
query="fighting movies"
qy_emb=model1.encode([query])
sims=[]
for e in tqdm(out_embs):
    sims.append(model1.similarity(qy_emb,e).numpy()[0])
df_ind=df_ind_mov[['id','title','description','release_year']].copy()
df_ind.loc[:,'sims']=np.array(sims)
df_ind.sort_values('sims',ascending=False)[:10]

100%|██████████| 570/570 [00:00<00:00, 5926.74it/s]


,id,title,description,release_year,sims
4295,tm818239,Penalty,A story of a North-east Indian football player...,2019,0.627975
2895,tm246330,Pandigai,A low-level hotel worker gets involved in a st...,2017,0.607773
1451,tm201365,Brothers,"David and Monty, estranged half-brothers, trai...",2015,0.605362
1016,tm167107,Saheb Biwi Aur Gangster Returns,"The Royal Scandal, the war for power and fight...",2013,0.594368
4565,tm471802,Sur Sapata,This story is about young boys who have capabi...,2019,0.587664
974,tm181092,Race 2,Ranveer Singh (Saif Ali Khan) travels to exoti...,2013,0.565047
2273,tm366590,Sanju,Sanju explores some of the most crucial chapte...,2018,0.562703
423,tm32885,Rang De Basanti,After a group of friends graduate from Delhi U...,2006,0.558188
2975,tm367613,Mr. Chandramouli,"Raghav, a professional boxer, finds himself in...",2018,0.553877
1076,tm159296,One by Two,This is the story of Amit and Samara - two sou...,2014,0.552326


In [26]:
query="patriotism"
qy_emb=model1.encode([query])
sims=[]
for e in tqdm(out_embs):
    sims.append(model1.similarity(qy_emb,e).numpy()[0])
df_ind=df_ind_mov[['id','title','description','release_year']].copy()
df_ind.loc[:,'sims']=np.array(sims)
df_ind.sort_values('sims',ascending=False)[:10]

100%|██████████| 570/570 [00:00<00:00, 6925.34it/s]


,id,title,description,release_year,sims
4565,tm471802,Sur Sapata,This story is about young boys who have capabi...,2019,0.543247
423,tm32885,Rang De Basanti,After a group of friends graduate from Delhi U...,2006,0.539653
4417,tm435836,Mere Pyare Prime Minister,A story about four children living in a Mumbai...,2019,0.535637
470,tm101609,Raajneeti,It is the story of a fiercely fought election ...,2010,0.526162
4295,tm818239,Penalty,A story of a North-east Indian football player...,2019,0.522644
5170,tm473502,83,"On the 25th of June 1983, the Lord’s Cricket G...",2021,0.521088
621,tm311516,Koi Aap Sa,"An emotional Mumbai-based football player, Roh...",2005,0.519422
182,tm184326,Damini,The theme revolves around the character Damini...,1993,0.511970
2719,tm319310,CIA: Comrade In America,A young Aji Mathew is a communist who goes the...,2017,0.509652
4472,tm832463,Mundina Nildana,Three different people with three different go...,2019,0.509261
